In [0]:
import os
#Creo la variable que contine el path del archivo .bacpac que contiene el backup de la base de datos
bacpac_path = "/Volumes/challenge/bronze_catalog/file/Testing_ETL.bacpac"
#Verificio que el archivo existe
print(os.path.exists(bacpac_path))



In [0]:
import zipfile
# Extraigo el archivo de la ruta definida 
with zipfile.ZipFile(bacpac_path, "r") as bacpac:
    files = bacpac.namelist()
#leo el contenido del archivo
for file in files:
    print(file)

In [0]:
import zipfile
import os

# 1. Defino la ruta donde voy a extraer el archivo
extract_path = "/tmp/Testing_ETL"

# 2. Creo el directorio si no existe (exist_ok=True evita error si ya existe)
os.makedirs(extract_path, exist_ok=True)

# 3. Abro el archivo .bacpac como archivo ZIP en modo lectura
with zipfile.ZipFile(bacpac_path, "r") as bacpac:
    # 4. Extraigo todo el contenido del ZIP al directorio extract_path
    bacpac.extractall(extract_path)

# 5. Imprimo la lista de archivos y carpetas en el primer nivel del directorio extraído
print(os.listdir(extract_path))

# 6. Recorro recursivamente todo el árbol de directorios extraído
for root, dirs, files in os.walk(extract_path):
    # 7. Para cada archivo encontrado, imprimo su ruta completa
    for file in files:
        print(os.path.join(root, file))

In [0]:
import glob
import os

bcp_files = sorted(
    glob.glob("/tmp/Testing_ETL/Data/dbo.Unificado/*.BCP")
)

print("Archivos BCP encontrados:", len(bcp_files))

for file in bcp_files:
    print(os.path.basename(file))

In [0]:
def parse_bcp_record(data, offset):
    # Define una función que parsea un registro BCP completo desde los datos binarios
    # Parámetros: data = bytes del archivo BCP, offset = posición actual de lectura
    # Retorna: lista de valores del registro y el nuevo offset
    
    values = []  # Crea una lista vacía para almacenar los 14 valores de columnas del registro

    # Columns 1-12
    # Lee las primeras 12 columnas que son campos de texto (UTF-16-LE con longitud precedente)
    for _ in range(12):
        # Llama a read_string_field para extraer un campo de texto y actualizar el offset
        value, offset = read_string_field(data, offset)
        # Agrega el valor de texto leído a la lista
        values.append(value)

    # Column 13: FECHA_COPIA
    # Lee la columna 13 que es un campo datetime (8 bytes de datos + 1 byte de longitud)
    value, offset = read_datetime_field(data, offset)
    # Agrega el valor datetime a la lista
    values.append(value)

    # Column 14: RESULTADO
    # Lee la columna 14 que es otro campo de texto
    value, offset = read_string_field(data, offset)
    # Agrega el último valor de texto a la lista
    values.append(value)

    # Retorna la lista completa de 14 valores y el offset actualizado (apunta al siguiente registro)
    return values, offset

In [0]:
import struct  # Importa el módulo struct para trabajar con datos binarios

def read_string_field(data, offset):
    # Leer longitud: 2 bytes, little-endian
    # struct.unpack_from extrae 2 bytes ("<H" = unsigned short, little-endian) desde offset
    # [0] obtiene el primer (y único) valor del resultado
    length = struct.unpack_from("<H", data, offset)[0]
    offset += 2  # Avanza el offset 2 bytes después de leer la longitud
    
    # Leer contenido UTF-16-LE
    # Extrae 'length' bytes desde la posición actual y los decodifica como texto UTF-16-LE
    value = data[offset:offset + length].decode("utf-16-le")
    offset += length  # Avanza el offset por la cantidad de bytes leídos
    
    return value, offset  # Retorna el valor de texto y el nuevo offset

# Abre el archivo BCP en modo lectura binaria
with open(bcp_path, "rb") as f:
    data = f.read()  # Lee todo el contenido del archivo en memoria

offset = 0  # Inicializa el offset en 0 (inicio del archivo)

values = []  # Lista vacía para almacenar los valores leídos

# Lee 12 campos de texto consecutivos del archivo BCP
for i in range(12):
    value, offset = read_string_field(data, offset)  # Lee un campo y actualiza offset
    values.append(value)  # Agrega el valor a la lista

# Imprime cada valor con su número de columna (1-12)
for i, value in enumerate(values, start=1):
    print(i, repr(value))  # repr() muestra el valor con comillas y caracteres especiales visibles

In [0]:
from datetime import datetime, timedelta

def read_datetime_field(data, offset):
    # 1. Lee el primer byte en la posición offset como la longitud del campo datetime
    length = data[offset]

    # 2. Valida que la longitud sea exactamente 8 bytes (formato datetime de SQL Server)
    if length != 8:
        raise ValueError(
            f"Expected 8 bytes for datetime, got {length}"
        )

    # 3. Extrae los 8 bytes del valor datetime (desde offset+1 hasta offset+9)
    raw = data[offset + 1:offset + 9]

    # 4. Convierte los primeros 4 bytes a un entero con signo (days desde 1900-01-01)
    days = int.from_bytes(raw[:4], "little", signed=True)
    
    # 5. Convierte los siguientes 4 bytes a un entero con signo (ticks de tiempo)
    ticks = int.from_bytes(raw[4:8], "little", signed=True)

    # 6. Construye el valor datetime sumando días y segundos a la fecha base 1900-01-01
    #    - Suma 'days' días a 1900-01-01
    #    - Suma 'ticks/300' segundos (ticks se divide por 300 para obtener segundos)
    value = (
        datetime(1900, 1, 1)
        + timedelta(days=days)
        + timedelta(seconds=ticks / 300)
    )

    # 7. Retorna el datetime calculado y el nuevo offset (avanzado 9 bytes: 1 de longitud + 8 de datos)
    return value, offset + 9

In [0]:
all_records = []

for bcp_path in bcp_files:

    with open(bcp_path, "rb") as f:
        data = f.read()

    offset = 0
    file_records = []

    while offset < len(data):
        record, offset = parse_bcp_record(data, offset)
        file_records.append(record)

    print(
        f"{os.path.basename(bcp_path)} -> "
        f"{len(file_records)} registros"
    )

    all_records.extend(file_records)

print("================================")
print("TOTAL DE REGISTROS:", len(all_records))

In [0]:
columns = [
    "CHROM",
    "POS",
    "ID",
    "REF",
    "ALT",
    "QUAL",
    "FILTER",
    "INFO",
    "FORMAT",
    "MUESTRA",
    "VALOR",
    "ORIGEN",
    "FECHA_COPIA",
    "RESULTADO"
]
df_origen = spark.createDataFrame(all_records, columns)
df_origen.createOrReplaceTempView("unificado_origen")

print("Registros:", df_origen.count())
print("Columnas:", len(df_origen.columns))

Crea la tabla de datos crudos


In [0]:
%sql
CREATE TABLE IF NOT EXISTS challenge.bronze_catalog.unificado
(CHROM string,
POS string,
ID string,
REF string,
ALT string,
QUAL string,
FILTER string,
INFO string,
FORMAT string,
MUESTRA string,
VALOR string,
ORIGEN string ,
FECHA_COPIA timestamp,
RESULTADO string)
USING DELTA

In [0]:
%sql
--DROP TABLE IF EXISTS challenge.bronze_catalog.unificado

Inserta en la tabla de datos crudos a partir de la vista del dataframe

In [0]:
%sql
INSERT INTO challenge.bronze_catalog.unificado
SELECT
    CHROM,
    POS,
    ID,
    REF,
    ALT,
    QUAL,
    FILTER,
    INFO,
    FORMAT,
    MUESTRA,
    VALOR,
    ORIGEN,
    FECHA_COPIA,
    RESULTADO
FROM unificado_origen;

In [0]:
%sql
select * from challenge.bronze_catalog.unificado

In [0]:
%sql
select count(*) from challenge.bronze_catalog.unificado